# **2.4. Parameter-Efficient Fine-Tuning (PEFT) : LoRA Fine-tuning**


## **0. Import Library**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from google.colab import files

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType

from sklearn.model_selection import train_test_split
from sklearn.metrics import *

## **1. Load dataset**

* select the provided [company-document-text.csv] file.

In [ ]:
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

In [ ]:
data_path

In [ ]:
data = pd.read_csv(data_path)
data.head()

In [ ]:
sns.countplot(x='label', data = data, hue = 'label')
plt.grid()
plt.show()

In [ ]:
sns.histplot(x='word_count', data = data, bins = 50)
plt.grid()
plt.show()

## **2. Pre-processing Data**

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
data['label'] = le.fit_transform(data['label'])
data.head()

In [ ]:
data2 = data.loc[:, ['text','label']].copy()

In [ ]:
train, val = train_test_split(data2, test_size=0.3, random_state=10)

In [ ]:
# tensor dataset
train_ds = Dataset.from_pandas(train)
val_ds = Dataset.from_pandas(val)

In [ ]:
train_ds[:3]

## **3. Tokenization**

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def preprocess_function(data):
    return tokenizer(data['text'], padding="max_length", truncation=True, max_length=100)

train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)

## **4. Load Pre-trained Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 4)

## **5. Set Training Arguments**

In [ ]:
# TrainingArguments
args = TrainingArguments(
    output_dir = './results',
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate = 2e-4,               # LoRA는 조금 큰 학습률
    per_device_train_batch_size = 32,   # 학습 배치 사이즈
    per_device_eval_batch_size = 32,
    num_train_epochs = 3,               # 에폭 수
    weight_decay = 0.02,                # weight decay
    load_best_model_at_end = True,      # earlystopping 사용하기 위해 필요
    logging_dir ='./logs',
    logging_steps = 10,
    report_to="tensorboard"
)

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,      # sequence classification
    r=8,                             # rank (smaller → fewer trainable params)
    lora_alpha=32,                   # scaling factor (too small: weak; too large: unstable)
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"],  # attention layers to target (BERT)
    bias="none"                      # do not train biases
)

# Apply LoRA (train only adapter layers)
model_lora = get_peft_model(model, lora_config)

## **6. Fine-tune Model**

In [ ]:
trainer = Trainer(
    model=model_lora,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
want_keys = ["input_ids", "attention_mask"]
batch = {k: torch.tensor(val_ds[k]) for k in want_keys}
batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)   # works
logits = outputs.logits        # [N, num_labels]
pred  = logits.argmax(dim=-1)  # 예측 라벨 id

In [ ]:
pred = pred.cpu().numpy()

In [ ]:
print(confusion_matrix(val_ds['label'], pred))
print(classification_report(val_ds['label'], pred, target_names = le.classes_))